In [ ]:
# ==============================================
# Mini Project : Association Rule Mining ตัวอย่างสำหรับคำถามข้อ 3
# 1) ขั้นตอน Support + Confidence ให้กฎจำนวนมาก แต่มี spurious rules
# 2) การใช้ Lift ช่วยกำจัดกฎที่เกิดจาก popularity ของ item
# 3) การใช้ p-value ช่วยยืนยันว่าความสัมพันธ์มีนัยสำคัญทางสถิติ
# ส่งผลให้จำนวนกฎลดลง แต่มีความน่าเชื่อถือมากขึ้น
# ==============================================

import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import chi2_contingency

pd.set_option('display.max_columns', None)   # แสดงทุก column
pd.set_option('display.width', 1000)         # กำหนดความกว้างหน้าจอ
pd.set_option('display.expand_frame_repr', False)  # ไม่ wrap ข้อความในบรรทัด

# ==============================================
# 1) LOAD DATA
# ==============================================

url = "https://raw.githubusercontent.com/pakornlee/ml_example/23665225ce5781e8ea782e18829e6108a5a4c92f/transactions_1000_onehot.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.strip()
df = df.astype(int)

min_sup = 0.05
min_conf = 0.3
top_n = 30

# ==============================================
# 2) COMPACT (Bitset + Pruning + Top-N)
# ==============================================

def run_compact(df, min_sup, max_k=6, top_n=30):
    total = len(df)

    bit_data = {}
    for col in df.columns:
        bitstring = ''.join(df[col].astype(str))
        bit_data[frozenset([col])] = int(bitstring, 2)

    L = {1: {}}

    # L1
    for itemset, bit in bit_data.items():
        sup = bin(bit).count('1') / total
        if sup >= min_sup:
            L[1][itemset] = (bit, sup)

    # Top-N
    L[1] = dict(sorted(L[1].items(), key=lambda x: x[1][1], reverse=True)[:top_n])

    # Lk
    for k in range(2, max_k + 1):
        L[k] = {}
        prev = list(L[k-1].keys())

        candidates = []
        for i in range(len(prev)):
            for j in range(i+1, len(prev)):
                union = prev[i] | prev[j]
                if len(union) == k:
                    candidates.append(union)

        # prune
        for c in candidates:
            if all((c - frozenset([x])) in L[k-1] for x in c):
                items = list(c)
                bit = bit_data[frozenset([items[0]])]

                for item in items[1:]:
                    bit &= bit_data[frozenset([item])]

                sup = bin(bit).count('1') / total
                if sup >= min_sup:
                    L[k][c] = (bit, sup)

        if len(L[k]) == 0:
            break

        L[k] = dict(sorted(L[k].items(), key=lambda x: x[1][1], reverse=True)[:top_n])

    results = {}
    for k in L:
        for itemset, (_, sup) in L[k].items():
            results[itemset] = sup

    return results

# ==============================================
# 3) GENERATE ASSOCIATION RULES
# ==============================================

def generate_rules(freq_itemsets, df):
    rules = []
    total = len(df)

    for itemset, sup in freq_itemsets.items():
        if len(itemset) < 2:
            continue

        for i in range(1, len(itemset)):
            for antecedent in combinations(itemset, i):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent

                sup_X = freq_itemsets.get(antecedent, 0)
                sup_Y = freq_itemsets.get(consequent, 0)

                if sup_X == 0 or sup_Y == 0:
                    continue

                confidence = sup / sup_X
                lift = confidence / sup_Y

                # chi-square → p-value
                X = df[list(antecedent)].all(axis=1)
                Y = df[list(consequent)].all(axis=1)

                n11 = ((X) & (Y)).sum()
                n10 = ((X) & (~Y)).sum()
                n01 = ((~X) & (Y)).sum()
                n00 = ((~X) & (~Y)).sum()

                table = [[n11, n10], [n01, n00]]

                try:
                    chi2, p, _, _ = chi2_contingency(table)
                except:
                    p = 1

                rules.append({
                    'antecedent': antecedent,
                    'consequent': consequent,
                    'support': sup,
                    'confidence': confidence,
                    'lift': lift,
                    'p_value': p
                })

    return pd.DataFrame(rules)

# ==============================================
# 4) RUN
# ==============================================

freq_itemsets = run_compact(df, min_sup)
rules = generate_rules(freq_itemsets, df)

print("All Rules:", len(rules))

# ==============================================
# 5) FILTER 3 APPROACHES
# ==============================================

# (A) Support + Confidence
rules_sc = rules[
    (rules['support'] >= min_sup) &
    (rules['confidence'] >= min_conf)
].sort_values(['confidence', 'support'], ascending=False)

# (B) + Lift
rules_scl = rules_sc[
    (rules_sc['lift'] > 1)
].sort_values(['lift', 'confidence'], ascending=False)

# (C) + p-value
rules_sclp = rules_scl[
    (rules_scl['p_value'] < 0.05)
].sort_values(['p_value', 'lift'], ascending=[True, False])

# ==============================================
# 6) SHOW RESULT
# ==============================================

print("\n===== Basic Rules (A) = Support + Confidence =====")
print(rules_sc.head(10).to_string(index=False))

print("\n===== Strong / Interesting Rules (B) = (A) + Lift =====")
print(rules_scl.head(10).to_string(index=False))

print("\n===== Significant Rules (C) = (B) + p-value =====")
print(rules_sclp.head(50).to_string(index=False))

print("\nCOUNT COMPARISON")
print(f"(A) Support & Cofidence : {len(rules_sc)} rules")
print(f"(B) Support, Confidence & Lift : {len(rules_scl)} rules")
print(f"(C) Support, Confidence, Lift & p-value : {len(rules_sclp)} rules")

# ==============================================
# 7) IDENTIFY SPURIOUS RULES
# ==============================================

# (1) ถูกตัดออกในขั้นตอน B (Lift <= 1)
spurious_lift = rules_sc[
    (rules_sc['lift'] <= 1)
].copy()

spurious_lift['reason'] = '     Low Lift (<=1) → Independent'

# (2) ถูกตัดออกในขั้นตอน C (p-value >= 0.05)
spurious_p = rules_scl[
    (rules_scl['p_value'] >= 0.05)
].copy()

spurious_p['reason'] = '     High p-value (>=0.05) → Not Significant'

# ==============================================
# 8) SHOW SPURIOUS RULES
# ==============================================

print("\n🔴 Spurious Rules removed in Step B (Lift filter)")
print(spurious_lift.sort_values(['lift'], ascending=False).head(10).to_string(index=False))

print("\n🔴 Spurious Rules removed in Step C (p-value filter)")
print(spurious_p.sort_values(['p_value'], ascending=True).head(10).to_string(index=False))

# ==============================================
# 9) COMBINE SPURIOUS RULES FROM B AND C
# ==============================================

spurious_all = pd.concat([spurious_lift, spurious_p])

print("\n===== SUMMARY OF SPURIOUS RULES =====")
print(spurious_all[['antecedent','consequent','support','confidence','lift','p_value','reason']].head(20).to_string(index=False))


All Rules: 224

===== Basic Rules (A) = Support + Confidence =====
          antecedent consequent  support  confidence     lift       p_value
            (cereal)     (milk)    0.192    1.000000 3.134796 1.725405e-111
             (bread)     (milk)    0.129    1.000000 3.134796  5.967343e-70
       (cereal, bag)     (milk)    0.129    1.000000 3.134796  5.967343e-70
             (chips)     (beer)    0.113    1.000000 4.032258  5.139122e-85
        (bread, bag)     (milk)    0.087    1.000000 3.134796  2.081941e-45
     (cereal, water)     (milk)    0.083    1.000000 3.134796  3.476111e-43
(bag, cereal, water)     (milk)    0.056    1.000000 3.134796  1.174017e-28
        (beer, rice)      (bag)    0.080    0.747664 1.072688  2.733135e-01
        (soda, rice)      (bag)    0.081    0.743119 1.066168  3.175103e-01
     (chicken, soda)      (bag)    0.066    0.741573 1.063950  4.021196e-01

===== Strong / Interesting Rules (B) = (A) + Lift =====
   antecedent    consequent  support  co